In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys

sys.path.append('../src')

import util as ut

In [ ]:
df_experiment = pd.read_csv("../data/raw/US_Accidents_March23.csv")

In [ ]:
df_experiment.shape

In [ ]:
df_experiment.describe()

In [ ]:
df_experiment.info()

#### Show only null columns

In [ ]:
missing_data = df_experiment.isnull().sum()[df_experiment.isnull().sum() > 0].sort_values(ascending=False)

print("--- Number of Null ---")
print(missing_data)

In [ ]:
missing_percent = (df_experiment.isnull().sum() / len(df_experiment)) * 100
missing_df = missing_percent[missing_percent > 0].sort_values(ascending=False)

print("--- Percent of Null ---")
print(missing_df)

#### Drop duplicate rows

In [ ]:
df_experiment = df_experiment.drop_duplicates()

#### Drop the lines that contain null values ​​in those columns.

In [ ]:
# Threshole 2 %
threshold_percent = 2.0
cols_to_dropna = missing_percent[(missing_percent > 0) & (missing_percent <= threshold_percent)].index.tolist()

print("--- Columns that contain less than 2% null ---")
print(cols_to_dropna)

In [ ]:
rows_before = df_experiment.shape[0]
df_experiment = df_experiment.dropna(subset=cols_to_dropna)
    
rows_after = df_experiment.shape[0]
rows_dropped = rows_before - rows_after
    
print(f"Number of rows delete: {rows_dropped:,} rows ({(rows_dropped/rows_before)*100:.2f}%)")
print(f"Number of rows remaining: {rows_after:,} rows")

#### Change time format

In [ ]:
df_experiment["Start_Time"] = pd.to_datetime(df_experiment["Start_Time"], errors="coerce")
df_experiment["End_Time"] = pd.to_datetime(df_experiment["End_Time"], errors="coerce")

# Target Column
df_experiment["Duration(min)"] = (df_experiment["End_Time"] - df_experiment["Start_Time"]).dt.total_seconds() / 60.0

In [ ]:
df_experiment.dropna(subset=["Start_Time", "End_Time", "Duration(min)"], inplace=True)

#### Check Nagative Duration 

In [ ]:
print(f"Negative durations: {(df_experiment['Duration(min)'] < 0).sum()} rows")

#### Check Unique Values for Categorical Columns

In [ ]:
categorical_columns = df_experiment.select_dtypes(include=['object', 'bool']).columns

print(f"--- Unique Values in Categorical Columns ({len(categorical_columns)} columns) ---\n")
for col in categorical_columns:
    n_unique = df_experiment[col].nunique()
    print(f"{col:25} | Unique: {n_unique:10,}")

    if n_unique <= 20:
        print(f"Values: {df_experiment[col].unique()}")
    print("-" * 50)

#### Check Wind direction

In [ ]:
df_experiment["Wind_Direction"].unique()

In [ ]:
df_experiment['Wind_Direction'] = df_experiment['Wind_Direction'].replace({
    'North': 'N',
    'South': 'S',
    'East': 'E',
    'West': 'W',
    'Variable': 'VAR',
    'CALM': 'Calm'
})

print(f"Unique Wind Direction after fix: {df_experiment['Wind_Direction'].unique()}")

#### Consistency Check (County & City)

In [ ]:
print("--- Shortest County Names ---")
county_names = df_experiment['County'].unique().astype(str)
shortest_counties = sorted(county_names, key=len)[:20]
print(shortest_counties)

In [ ]:
print("--- Shortest City Names ---")
county_names = df_experiment['City'].unique().astype(str)
shortest_counties = sorted(county_names, key=len)[:20]
print(shortest_counties)

#### Drop Column with has 1 unique value

In [ ]:
df_experiment.drop(["Country", "Turning_Loop"], axis=1, inplace=True)

#### Check Outlier

In [ ]:
ut.skewed(df_experiment)

In [ ]:
# Distance
plt.boxplot(df_experiment["Distance(mi)"])
plt.title("Boxplot showing outliers Distances")
plt.show()

df_experiment["Distance(mi)"].quantile([0.95, 0.99, 0.999])

In [ ]:
# Precipitation(in)
plt.boxplot(df_experiment["Precipitation(in)"].dropna())
plt.title("Boxplot showing outliers Precipitation(in)")
plt.show()

df_experiment["Precipitation(in)"].quantile([0.95, 0.99, 0.999])

In [ ]:
# Wind Speed
plt.boxplot(df_experiment["Wind_Speed(mph)"].dropna())
plt.title("Boxplot showing outliers Wind_Speed(mph)")
plt.show()

df_experiment["Wind_Speed(mph)"].quantile([0.95, 0.99, 0.999])

In [ ]:
# Duration
plt.boxplot(df_experiment["Duration(min)"])
plt.title("Boxplot showing outliers Duration")
plt.show()

df_experiment["Duration(min)"].quantile([0.95, 0.99, 0.999])

In [ ]:
# Pressure
plt.boxplot(df_experiment["Pressure(in)"].dropna())
plt.title("Boxplot showing outliers Pressure")
plt.show()

df_experiment["Pressure(in)"].quantile([0.95, 0.99, 0.999])

In [ ]:
# Visibility
plt.boxplot(df_experiment["Visibility(mi)"].dropna())
plt.title("Boxplot showing outliers Visibility")
plt.show()

df_experiment["Visibility(mi)"].quantile([0.95, 0.99, 0.999])

In [ ]:
# Visibility
plt.boxplot(df_experiment["Severity"])
plt.title("Boxplot showing outliers Severity")
plt.show()

df_experiment["Severity"].quantile([0.95, 0.99, 0.999])

#### Manage excessively Outlier

In [ ]:
outlier_columns = ["Distance(mi)", "Pressure(in)", "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)", "Duration(min)"]

df_outlier = ut.manage_outlier(df_experiment, outlier_columns=outlier_columns)

In [ ]:
len(df_outlier)

In [ ]:
cols_to_drop = [
    "ID", "Source",
    "End_Lat", "End_Lng",
    "Timezone", "Airport_Code", "Weather_Timestamp",
    "End_Time"
]

df_basic_clean = df_outlier.drop(columns=cols_to_drop, errors="ignore")

#### Export to CSV

In [ ]:
print(f"Number of rows: {df_basic_clean.shape[0]:,} rows")

df_basic_clean.to_csv("../data/processed/01.1/accidents_basic_clean.csv", index=False)